# Análise de Dados de Mensagens do Telegram com DuckDB

**Disciplina:** CKP9011 - Introdução à Ciência de Dados / CK0223 - Mineração de Dados

**Instituição:** Universidade Federal do Ceará (UFC)

**Atividade:** Lista de Exercícios 2 - Análise com DuckDB

## 1. Introdução

Este notebook documenta o processo de limpeza, preparação e análise de dados do dataset `fakeTelegram.BR_2022.csv`. O objetivo é responder a 25 questões analíticas utilizando **DuckDB**, um banco de dados analítico (OLAP) de alta performance.

O fluxo de trabalho consistirá em:
1.  Carregar e limpar os dados brutos usando Pandas.
2.  Exportar os dados limpos para o formato Parquet.
3.  Registrar o arquivo Parquet no DuckDB.
4.  Executar todas as 25 consultas analíticas diretamente em SQL.

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import re

In [ ]:
file = pd.read_csv('../data/raw/fakeTelegram.BR_2022.csv')

## 2. Limpeza e Preparação dos Dados (Pandas)

Antes de analisar, preparamos os dados.

### 2.1. (Item b) Remoção de "Trava-Zaps"

Primeiro, removemos mensagens maliciosas (trava-zaps) usando a função de heurística desenvolvida anteriormente.

In [ ]:
trava_zaps = file[file['trava_zap'] == True].index

file.drop(trava_zaps, inplace=True)

### 2.2. (Item c) Exportar os dados para um arquivo Parquet

Com os dados limpos, os salvamos em formato Parquet. Parquet é um formato colunar otimizado para análises, sendo muito mais rápido e eficiente em espaço do que o CSV.

In [ ]:
file.to_parquet('../data/processed/clean_file.parquet', engine='fastparquet')

df_parquet = pd.read_parquet('../data/processed/clean_file.parquet')

### 2.3. (Item d) Exportar os dados para o DuckDB

Agora, preparamos o ambiente de análise. Utilizando o DuckDB, registramos nosso DataFrame limpo (`df_parquet`) como uma tabela virtual chamada `telegram_data`. Isso nos permite executar consultas SQL de alta performance diretamente nos dados em memória, sem duplicação.

In [ ]:
con = duckdb.connect()

con.register('telegram_data', df_parquet)

## 3. Análises com DuckDB

Iniciamos as 25 consultas analíticas.

### 3.1. A quantidade de mensagens
*Usamos `COUNT(*)` para contar o número total de linhas (mensagens) na tabela.*

In [ ]:
query_text = """
SELECT COUNT(*)
FROM telegram_data
"""

con.query(query_text).to_df()

### 3.2. A quantidade de usuários
*Usamos `COUNT(DISTINCT ...)` para contar o número de IDs de usuários únicos.*

In [ ]:
query_text = """
SELECT COUNT(DISTINCT id_member_anonymous)
FROM telegram_data
"""

con.query(query_text).to_df()

### 3.3. A quantidade de grupos
*Similarmente, contamos o número de IDs de grupos únicos.*

In [ ]:
query_text = """
SELECT COUNT(DISTINCT id_group_anonymous)
FROM telegram_data
"""

con.query(query_text).to_df()

### 3.4. Quantidade de mensagens que possuem apenas texto
*Filtramos (`WHERE`) mensagens onde o `media_type` é `NULL` (vazio) ou contém a string 'nan', que é um artefato comum da conversão de dados ausentes.*

In [ ]:
query_text = """
SELECT COUNT(*)
FROM telegram_data
WHERE media_type IS NULL 
  OR lower(media_type) = 'nan'
"""

con.query(query_text).to_df()

### 3.5. Quantidade de mensagens contendo mídias
*Esta é a consulta oposta à anterior. Usamos `AND` para selecionar linhas onde `media_type` NÃO é `NULL` E também NÃO é a string 'nan'.*

In [ ]:
query_text = """
SELECT COUNT(*)
FROM telegram_data
WHERE media_type IS NOT NULL 
  AND lower(media_type) != 'nan'
"""

con.query(query_text).to_df()

### 3.6. Quantidade de mensagens por tipo de mídia (jpg, mp4 etc)
*Agrupamos (`GROUP BY`) as mídias válidas por seu tipo e contamos as ocorrências em cada grupo.*

In [ ]:
query_text = """
SELECT media_type, COUNT(*) 
FROM telegram_data 
WHERE media_type IS NOT NULL 
  AND lower(media_type) != 'nan'
GROUP BY media_type
"""

con.query(query_text).to_df()

### 3.12. As 30 URLs que mais se repetem (mais compartilhadas)
*Agrupamos por `media_url`, contamos (`COUNT`), ordenamos em ordem decrescente (`DESC`) e pegamos as 30 primeiras (`LIMIT 30`) para encontrar as URLs mais frequentes.*

In [ ]:
query_text = """
SELECT
    media_url,
    COUNT(*) AS contagem
FROM telegram_data
WHERE media_url IS NOT NULL AND lower(media_url) != 'nan'
GROUP BY media_url
ORDER BY contagem DESC
LIMIT 30
"""
con.query(query_text).to_df()

### 3.13. Os 30 domínios que mais se repetem (mais compartilhados)
*Usamos a função `regexp_extract` para extrair o domínio de cada URL. Para garantir que 't.me' e 'T.me' sejam contados juntos, usamos `lower()` antes de agrupar. Filtramos o resultado com `HAVING` para remover domínios vazios que o regex não conseguiu capturar.*

In [ ]:
query_text = r"""
SELECT
    lower(regexp_extract(media_url, '^(?:https?://)?(?:www\.)?([^/]+)', 1)) AS dominio,
    COUNT(*) AS contagem
FROM telegram_data
WHERE media_url IS NOT NULL AND lower(media_url) != 'nan'
GROUP BY dominio
HAVING dominio != '' 
ORDER BY contagem DESC
LIMIT 30
"""

con.query(query_text).to_df()

### 3.14. Os 30 usuários mais ativos
*Agrupamos por `id_member_anonymous` e contamos as mensagens. Adicionamos `WHERE id_member_anonymous IS NOT NULL` para remover o grupo "None" (usuários não identificados) do ranking.*

In [ ]:
query_text = """
SELECT 
    id_member_anonymous, 
    COUNT(*) AS total_mensagens 
FROM telegram_data 
WHERE id_member_anonymous IS NOT NULL
GROUP BY id_member_anonymous 
ORDER BY total_mensagens DESC 
LIMIT 30
"""

con.query(query_text).to_df()


### 3.15. Os 30 usuários que mais compartilharam texto
*Mesma lógica do item 14, mas adicionamos um filtro `WHERE` para incluir apenas mensagens de texto (onde `media_type` é nulo ou 'nan').*

In [ ]:
query_text = """
SELECT
    id_member_anonymous,
    COUNT(*) AS contagem_texto
FROM telegram_data
WHERE
    (media_type IS NULL OR lower(media_type) = 'nan')
    AND id_member_anonymous IS NOT NULL
GROUP BY id_member_anonymous
ORDER BY contagem_texto DESC
LIMIT 30
"""

con.query(query_text).to_df()

### 3.16. Os 30 usuários que mais compartilharam mídias
*Novamente, a mesma lógica, mas com o filtro `WHERE` para mídias válidas (onde `media_type` não é nulo e não é 'nan').*

In [ ]:
query_text = """
SELECT
    id_member_anonymous,
    COUNT(*) AS contagem_midia
FROM telegram_data
WHERE
    (media_type IS NOT NULL AND lower(media_type) != 'nan')
    AND id_member_anonymous IS NOT NULL
GROUP BY id_member_anonymous
ORDER BY contagem_midia DESC
LIMIT 30
"""

con.query(query_text).to_df()

### 3.17. As 30 mensagens mais compartilhadas
*Agrupamos pelo conteúdo do texto (`text_content_anonymous`) para encontrar as mensagens virais mais comuns, filtrando textos nulos.*

In [ ]:
query_text = """
SELECT
    text_content_anonymous,
    COUNT(*) AS contagem
FROM telegram_data
WHERE text_content_anonymous IS NOT NULL
GROUP BY text_content_anonymous
ORDER BY contagem DESC
LIMIT 30
"""

con.query(query_text).to_df()

### 3.18. As 30 mensagens mais compartilhadas em grupos diferentes
*Esta é uma medida de "dispersão" (spread). Usamos `COUNT(DISTINCT id_group_anonymous)` para contar em quantos grupos únicos cada mensagem apareceu.*

In [ ]:
query_text = """
SELECT
    text_content_anonymous,
    COUNT(DISTINCT id_group_anonymous) AS contagem_grupos
FROM telegram_data
WHERE text_content_anonymous IS NOT NULL
GROUP BY text_content_anonymous
ORDER BY contagem_grupos DESC
LIMIT 30
"""

con.query(query_text).to_df()

### 3.19. Mensagens idênticas compartilhadas pelo mesmo usuário
*Agrupamos por duas colunas (usuário e texto) e usamos `HAVING COUNT(*) > 1` para filtrar apenas os pares (usuário, mensagem) que apareceram mais de uma vez.*

In [ ]:
query_text = """
SELECT
    id_member_anonymous,
    text_content_anonymous,
    COUNT(*) AS contagem
FROM telegram_data
WHERE id_member_anonymous IS NOT NULL
  AND text_content_anonymous IS NOT NULL
GROUP BY id_member_anonymous, text_content_anonymous
HAVING contagem > 1
ORDER BY contagem DESC
"""

con.query(query_text).to_df().head(15)

### 3.20. Mensagens idênticas compartilhadas pelo mesmo usuário em grupos distintos
*Combinamos as lógicas anteriores: Agrupamos por usuário e texto, contamos os grupos distintos (`COUNT(DISTINCT ...)`) e filtramos (`HAVING`) onde essa contagem de grupos é maior que 1.*

In [ ]:
query_text = """
SELECT
    id_member_anonymous,
    text_content_anonymous,
    COUNT(DISTINCT id_group_anonymous) AS contagem_grupos
FROM telegram_data
WHERE id_member_anonymous IS NOT NULL
  AND text_content_anonymous IS NOT NULL
GROUP BY id_member_anonymous, text_content_anonymous
HAVING contagem_grupos > 1
ORDER BY contagem_grupos DESC
"""

con.query(query_text).to_df().head(15)

### 3.21. Os 30 unigramas, bigramas e trigramas mais compartilhados
*Esta tarefa de Processamento de Linguagem Natural (NLP) é muito complexa para SQL. A abordagem correta é realizá-la no Pandas usando a biblioteca Scikit-learn. Definimos uma função que usa `CountVectorizer` para tokenizar, contar e rankear os N-gramas (sequências de 1, 2 ou 3 palavras).*

### Nota de Implementação (Relato Pessoal sobre o Item 21)

Esta foi, sem dúvida, a tarefa mais desafiadora da lista, pois migra da análise SQL tradicional para o Processamento de Linguagem Natural (NLP).

**Decisão de Abordagem:**
A extração de N-gramas (unigramas, bigramas, trigramas) poderia ser tentada via SQL (extremamente complexo) ou via Pandas puro (usando `.apply()`, `.explode()` e `.value_counts()`). A abordagem final escolhida foi utilizar a biblioteca `CountVectorizer` do Scikit-learn.

**Justificativa (Performance vs. Método):**
Inicialmente, foi feita uma tentativa de resolver o problema com Pandas puro, utilizando `.apply()`, mas a operação **esgotou a memória RAM** do sistema e falhou. Isso acontece porque métodos como `.explode()` tentam criar DataFrames intermediários massivos (milhões de N-gramas), o que não é escalável para um dataset deste tamanho.

O `CountVectorizer`, por outro lado, é a ferramenta padrão da indústria para esta tarefa. Ele é escrito em C e usa matrizes esparsas, sendo otimizado para tokenizar e contar milhões de palavras de forma eficiente em termos de memória e velocidade. Embora o processamento no dataset completo leve um tempo perceptível (cerca de 40-50 segundos), isso é, na verdade, um sinal de sua alta performance, já que a alternativa com `.apply()` falhou por falta de memória.

**Observação sobre *Stop Words*:**
Decidi **não** remover as *stop words* (palavras comuns como 'o', 'a', 'de', 'que'). A remoção é comum para *modelagem* de tópicos, mas a questão pede as 30 sequências "mais compartilhadas", sejam elas quais forem. Manter as *stop words* nos permite capturar frases comuns e conectivos reais do português (como "pra que", "é o", "da julia"), o que responde mais fielmente à pergunta da lista.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

textos = file['text_content_anonymous'].dropna()

def get_ngrams(textos, ngram_range, n=30):
   
    # ngram_range: (min, max) tipo: (1, 1) para unigramas, (2, 2) para bigramas.
    vec = CountVectorizer(ngram_range=ngram_range, lowercase=True) 
    X = vec.fit_transform(textos)
    
    soma_palavras = X.sum(axis=0) 
    
    palavras_freq = [(palavra, soma_palavras[0, idx]) for palavra, idx in vec.vocabulary_.items()]
    
    palavras_freq = sorted(palavras_freq, key = lambda x: x[1], reverse=True)
    
    return pd.DataFrame(palavras_freq[:n], columns=['Ngram', 'Frequencia'])

df_unigramas = get_ngrams(textos, ngram_range=(1, 1), n=30)
df_bigramas = get_ngrams(textos, ngram_range=(2, 2), n=30)
df_trigramas = get_ngrams(textos, ngram_range=(3, 3), n=30)


In [ ]:
df_unigramas

In [ ]:
df_bigramas

In [ ]:
df_trigramas

### 3.22. As 30 mensagens mais positivas (distintas)
*Como a coluna de sentimento (`score_sentiment`) já existe, podemos usá-la. Selecionamos 30 textos únicos (`DISTINCT`) e ordenamos pelo score em ordem decrescente (`DESC`) para pegar os mais positivos.*

In [ ]:
query_text = """
SELECT
    DISTINCT text_content_anonymous,
    score_sentiment
FROM telegram_data
WHERE text_content_anonymous IS NOT NULL
ORDER BY score_sentiment DESC
LIMIT 30
"""

con.query(query_text).to_df()

### 3.23. As 30 mensagens mais negativas (distintas)
*O oposto do item anterior. Ordenamos o score em ordem ascendente (`ASC`) para pegar os 30 mais negativos.*

In [ ]:
query_text = """
SELECT
    DISTINCT text_content_anonymous,
    score_sentiment
FROM telegram_data
WHERE text_content_anonymous IS NOT NULL
ORDER BY score_sentiment ASC
LIMIT 30
"""

con.query(query_text).to_df()

### 3.24. O usuário mais otimista
*Para encontrar o usuário mais otimista, agrupamos por usuário, calculamos a média do score (`AVG(score_sentiment)`) e encontramos o usuário com o maior score médio.*

In [ ]:
query_text = """
SELECT
    id_member_anonymous,
    AVG(score_sentiment) AS sentimento_medio
FROM telegram_data
WHERE id_member_anonymous IS NOT NULL
GROUP BY id_member_anonymous
ORDER BY sentimento_medio DESC
LIMIT 1
"""

con.query(query_text).to_df()

### 3.25. O usuário mais pessimista
*O oposto do item 24. Encontramos o usuário com o menor score médio de sentimento (`ORDER BY ... ASC`).*

In [ ]:
query_text = """
SELECT
    id_member_anonymous,
    AVG(score_sentiment) AS sentimento_medio
FROM telegram_data
WHERE id_member_anonymous IS NOT NULL
GROUP BY id_member_anonymous
ORDER BY sentimento_medio ASC
LIMIT 1
"""

con.query(query_text).to_df()

### 3.26. As 30 maiores mensagens
*Usamos a função `LENGTH()` do SQL para calcular o tamanho de cada texto e ordenamos (`ORDER BY ... DESC`) para encontrar os 30 maiores.*

In [ ]:
query_text = """
SELECT
    text_content_anonymous,
    LENGTH(text_content_anonymous) AS tamanho
FROM telegram_data
WHERE text_content_anonymous IS NOT NULL
ORDER BY tamanho DESC
LIMIT 30
"""

con.query(query_text).to_df()

### 3.27. As 30 menores mensagens
*Similar ao item 26, mas ordenamos de forma ascendente (`ASC`) e adicionamos um filtro `WHERE LENGTH(...) > 0` para garantir que não estamos apenas pegando strings vazias.*

In [ ]:
query_text = """
SELECT
    text_content_anonymous,
    LENGTH(text_content_anonymous) AS tamanho
FROM telegram_data
WHERE text_content_anonymous IS NOT NULL AND LENGTH(text_content_anonymous) > 0
ORDER BY tamanho ASC
LIMIT 30
"""

con.query(query_text).to_df()

### 3.28. O dia com a maior quantidade de mensagens
*Para agrupar por dia (ignorando a hora), usamos `CAST(date_message AS DATE)`. Em seguida, agrupamos, contamos, ordenamos e pegamos o primeiro resultado.*

In [ ]:
query_text = """
SELECT
    CAST(date_message AS DATE) AS dia,
    COUNT(*) AS contagem
FROM telegram_data
GROUP BY dia
ORDER BY contagem DESC
LIMIT 1
"""

con.query(query_text).to_df()

### 3.29. Mensagens com "FACÇÃO" e "CRIMINOSA"
*Usamos `ILIKE` (LIKE case-insensitive) com o operador `AND` para encontrar mensagens que contenham ambas as palavras-chave, em qualquer lugar do texto.*

In [ ]:
query_text = """
SELECT text_content_anonymous
FROM telegram_data
WHERE text_content_anonymous ILIKE '%FACÇÃO%'
  AND text_content_anonymous ILIKE '%CRIMINOSA%'
"""

con.query(query_text).to_df()

### 3.30. Mensagens com "SEGURANÇA"
*Usamos `ILIKE` para encontrar qualquer mensagem que contenha a palavra "segurança", ignorando maiúsculas ou minúsculas.*

In [ ]:
query_text = """
SELECT text_content_anonymous
FROM telegram_data
WHERE text_content_anonymous ILIKE '%SEGURANÇA%'
"""

con.query(query_text).to_df()